In [ ]:
!pip install rioxarray
!pip install rasterio
!pip install pystac_client
!pip install planetary_computer
!pip install pystac

In [ ]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data Science
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets
import xarray as xr

# Geospatial raster data handling
import rioxarray as rxr

# Geospatial data analysis
import geopandas as gpd

# Geospatial operations
import rasterio
from rasterio import windows  
from rasterio import features  
from rasterio import warp
from rasterio.warp import transform_bounds 
from rasterio.windows import from_bounds 

# Image Processing
from PIL import Image

# Coordinate transformations
from pyproj import Proj, Transformer, CRS

# Feature Engineering
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Planetary Computer Tools
import pystac_client
import planetary_computer as pc
from pystac.extensions.eo import EOExtension as eo

# Others
import os
from tqdm import tqdm

In [ ]:
# Load the training data from csv file and display the first few rows to inspect the data
ground_df = pd.read_csv("/kaggle/input/ey-opendata/Training_data_uhi_index_2025-02-18.csv")
ground_df.head()

In [ ]:
tiff_path = "/kaggle/input/ey-challenge/S2_sample.tiff"

if not os.path.exists(tiff_path):
    raise FileNotFoundError("GeoTIFF file not found!")

# Read the bands
with rasterio.open(tiff_path) as src:
    bands = [src.read(i) for i in range(1, 13)]

# Check if bands contain valid data
for i, band in enumerate(bands):
    print(f"Band {i+1} - Min: {band.min()}, Max: {band.max()}")

# Plotting
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()
titles = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]

for i, (band, title) in enumerate(zip(bands, titles)):
    if band.size == 0:
        print(f"Skipping {title} (empty band)")
        continue
    im = axes[i].imshow(band, cmap="viridis")
    axes[i].set_title(f"Band [{title}]")
    fig.colorbar(im, ax=axes[i])

plt.tight_layout()
plt.show()

In [ ]:
tiff_path = "/kaggle/input/landsat-tiff1/Landsat_LST.tiff"

if not os.path.exists(tiff_path):
    raise FileNotFoundError("GeoTIFF file not found!")

# Read the bands
with rasterio.open(tiff_path) as src:
    bands = [src.read(i) for i in range(1, 2)]

# Check if bands contain valid data
for i, band in enumerate(bands):
    print(f"Band {i+1} - Min: {band.min()}, Max: {band.max()}")

# Plotting
fig, ax = plt.subplots(figsize=(15, 10))  # Single Axes object
title = "lwir11"

if bands[0].size == 0:
    print(f"Skipping {title} (empty band)")
else:
    im = ax.imshow(bands[0], cmap="viridis")
    ax.set_title(f"Band [{title}]")
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Extracts satellite band values from a GeoTIFF based on coordinates from a csv file and returns them in a DataFrame.

def map_satellite_data(tiff_path, csv_path):
    
    # Load the GeoTIFF data
    data = rxr.open_rasterio(tiff_path)
    tiff_crs = data.rio.crs

    # Read the Excel file using pandas
    df = pd.read_csv(csv_path)
    latitudes = df['Latitude'].values
    longitudes = df['Longitude'].values

    # 3. Convert lat/long to the GeoTIFF's CRS
    # Create a Proj object for EPSG:4326 (WGS84 - lat/long) and the GeoTIFF's CRS
    proj_wgs84 = Proj(init='epsg:4326')  # EPSG:4326 is the common lat/long CRS
    proj_tiff = Proj(tiff_crs)
    
    # Create a transformer object
    transformer = Transformer.from_proj(proj_wgs84, proj_tiff)

    B01_values = []
    B02_values = []
    B03_values = []
    B04_values = []
    B05_values = []
    B06_values = []
    B07_values = []
    B08_values = []
    B8A_values = []
    B09_values = []
    B11_values = []
    B12_values = []

# Iterate over the latitudes and longitudes, and extract the corresponding band values
    for lat, lon in tqdm(zip(latitudes, longitudes), total=len(latitudes), desc="Mapping values"):
    # Assuming the correct dimensions are 'y' and 'x' (replace these with actual names from data.coords)
    
        B01_value = data.sel(x=lon, y=lat,  band=1, method="nearest").values
        B01_values.append(B01_value)
    
        B02_value = data.sel(x=lon, y=lat, band=2, method="nearest").values
        B02_values.append(B02_value)
        
        B03_value = data.sel(x=lon, y=lat, band=3, method="nearest").values
        B03_values.append(B03_value)
    
        B04_value = data.sel(x=lon, y=lat, band=4, method="nearest").values
        B04_values.append(B04_value)

        B05_value = data.sel(x=lon, y=lat, band=5, method="nearest").values
        B05_values.append(B05_value)

        B06_value = data.sel(x=lon, y=lat, band=6, method="nearest").values
        B06_values.append(B06_value)

        B07_value = data.sel(x=lon, y=lat, band=7, method="nearest").values
        B07_values.append(B07_value)

        B08_value = data.sel(x=lon, y=lat, band=8, method="nearest").values
        B08_values.append(B08_value)

        B8A_value = data.sel(x=lon, y=lat, band=9, method="nearest").values
        B8A_values.append(B8A_value)

        B09_value = data.sel(x=lon, y=lat, band=10, method="nearest").values
        B09_values.append(B09_value)

        B11_value = data.sel(x=lon, y=lat, band=11, method="nearest").values
        B11_values.append(B11_value)

        B12_value = data.sel(x=lon, y=lat, band=12, method="nearest").values
        B12_values.append(B12_value)

    # Create a DataFrame with the band values
    # Create a DataFrame to store the band values
    df = pd.DataFrame()
    df['B01'] = B01_values
    df['B02'] = B02_values
    df['B03'] = B03_values
    df['B04'] = B04_values
    df['B05'] = B05_values
    df['B06'] = B06_values
    df['B07'] = B07_values
    df['B08'] = B08_values
    df['B8A'] = B8A_values
    df['B09'] = B09_values
    df['B11'] = B11_values
    df['B12'] = B12_values
    
    return df

In [ ]:
# Mapping satellite data with training data.
final_data = map_satellite_data('/kaggle/input/ey-challenge/S2_sample.tiff', '/kaggle/input/ey-opendata/Training_data_uhi_index_2025-02-18.csv')

In [ ]:
final_data.head()

In [ ]:
from rasterio.warp import transform

def map_satellite_data2(tiff_path, csv_path):
    # Load the GeoTIFF file
    with rasterio.open(tiff_path) as src:
        tiff_crs = src.crs  # Get CRS of the raster

        # Read the CSV file
        df = pd.read_csv(csv_path)
        latitudes = df['Latitude'].values
        longitudes = df['Longitude'].values

        # Transform lat/lon coordinates to raster coordinate system
        x_coords, y_coords = transform('EPSG:4326', tiff_crs, longitudes, latitudes)

        # Read all bands
        num_bands = src.count
        band_values = {'lwir11': [] for i in range(num_bands)}

        # Extract values for each coordinate
        for x, y in tqdm(zip(x_coords, y_coords), total=len(x_coords), desc="Mapping values"):
            row, col = src.index(x, y)  # Get raster indices
            for i in range(1, num_bands + 1):
                try:
                    value = src.read(i)[row, col]  # Read pixel value from band
                except IndexError:
                    value = None  # Handle missing data
                band_values['lwir11'].append(value)

    # Convert to DataFrame
    df_bands = pd.DataFrame(band_values)
    
    return df_bands

In [ ]:
# Mapping satellite data with training data.
final_data2 = map_satellite_data2('/kaggle/input/landsat-tiff1/Landsat_LST.tiff', "/kaggle/input/ey-opendata/Training_data_uhi_index_2025-02-18.csv")

In [ ]:
final_data2.head()

In [ ]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2], axis=1)
    return data

In [ ]:
final_data = combine_two_datasets(final_data, final_data2)

In [ ]:
final_data.head()

In [ ]:
weather_data_path = '/kaggle/input/ey-challenge/NY_Mesonet_Weather.xlsx'
# Load weather data from both sheets
bronx_weather = pd.read_excel(weather_data_path, sheet_name="Bronx")
manhattan_weather = pd.read_excel(weather_data_path, sheet_name="Manhattan")

In [ ]:
bronx_weather

In [ ]:
from datetime import datetime
from geopy.distance import geodesic

# Define weather station locations
bronx_coords = (40.87248, -73.89352)
manhattan_coords = (40.76754, -73.96449)

# Convert datetime columns to datetime objects
ground_df['datetime'] = pd.to_datetime(ground_df['datetime'], format='%d-%m-%Y %H:%M')
bronx_weather['Date / Time'] = pd.to_datetime(bronx_weather['Date / Time'])
manhattan_weather['Date / Time'] = pd.to_datetime(manhattan_weather['Date / Time'])

# Function to determine the closest weather station
def get_closest_station(lat, lon):
    bronx_dist = geodesic((lat, lon), bronx_coords).km
    manhattan_dist = geodesic((lat, lon), manhattan_coords).km
    return 'bronx' if bronx_dist < manhattan_dist else 'manhattan'

# Function to find the closest weather record
def get_closest_weather_data(row):
    station = get_closest_station(row['Latitude'], row['Longitude'])
    weather_df = bronx_weather if station == 'bronx' else manhattan_weather
    
    closest_time_idx = (weather_df['Date / Time'] - row['datetime']).abs().idxmin()
    closest_weather = weather_df.loc[closest_time_idx, ['Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']]
    return closest_weather

# Apply function to df1
ground_df[['Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']] = ground_df.apply(get_closest_weather_data, axis=1)

# Display updated df1
ground_df.head()


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Function to parse KML and extract building centroids
def parse_kml(file_path):
    namespace = {"kml": "http://www.opengis.net/kml/2.2"}
    tree = ET.parse(file_path)
    root = tree.getroot()

    placemarks = root.findall(".//kml:Placemark", namespace)
    building_centroids = []

    for placemark in placemarks:
        coords_element = placemark.find(".//kml:coordinates", namespace)
        if coords_element is not None:
            coords_text = coords_element.text.strip()
            coords_list = [
                tuple(map(float, coord.split(",")[:2]))  # Extract lon, lat only
                for coord in coords_text.split()
            ]
            # Compute centroid
            centroid_lat = sum(p[1] for p in coords_list) / len(coords_list)
            centroid_lon = sum(p[0] for p in coords_list) / len(coords_list)
            building_centroids.append((centroid_lat, centroid_lon))

    return np.array(building_centroids)  # Convert to NumPy array for speed

# Fast Haversine distance calculation
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c  # Distance in meters

# Function to count buildings near UHI points (Parallelized)
def count_buildings_near_uhi(building_centroids, uhi_points, radius=80):
    def count_nearby_buildings(lat, lon):
        distances = haversine(lat, lon, building_centroids[:, 0], building_centroids[:, 1])
        return np.sum(distances <= radius)  # Count buildings within radius

    results = Parallel(n_jobs=-1)(
        delayed(count_nearby_buildings)(row['latitude'], row['longitude'])
        for _, row in tqdm(uhi_points.iterrows(), total=len(uhi_points), desc="Processing UHI Points")
    )

    uhi_points['building_count'] = results
    return uhi_points

# File paths
kml_file = "/kaggle/input/ey-challenge/Building_Footprint.kml"

# Parse data
building_centroids = parse_kml(kml_file)
uhi_data = ground_df

# Ensure correct column names
uhi_data.rename(columns={"Latitude": "latitude", "Longitude": "longitude"}, inplace=True)

# Count buildings near each UHI index point (optimized)
uhi_with_building_counts = count_buildings_near_uhi(building_centroids, uhi_data)

print(uhi_with_building_counts.head())


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Function to parse KML and extract building centroids
def parse_kml(file_path):
    namespace = {"kml": "http://www.opengis.net/kml/2.2"}
    tree = ET.parse(file_path)
    root = tree.getroot()

    placemarks = root.findall(".//kml:Placemark", namespace)
    building_centroids = []

    for placemark in placemarks:
        coords_element = placemark.find(".//kml:coordinates", namespace)
        if coords_element is not None:
            coords_text = coords_element.text.strip()
            coords_list = [
                tuple(map(float, coord.split(",")[:2]))  # Extract lon, lat only
                for coord in coords_text.split()
            ]
            # Compute centroid
            centroid_lat = sum(p[1] for p in coords_list) / len(coords_list)
            centroid_lon = sum(p[0] for p in coords_list) / len(coords_list)
            building_centroids.append((centroid_lat, centroid_lon))

    return np.array(building_centroids)  # Convert to NumPy array for speed

# Fast Haversine distance calculation
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c  # Distance in meters

# Function to count buildings in directional quadrants
def count_buildings_directional(building_centroids, uhi_points, radius=80):
    def check_directions(lat, lon):
        distances = haversine(lat, lon, building_centroids[:, 0], building_centroids[:, 1])
        nearby_buildings = building_centroids[distances <= radius]  # Filter buildings within the radius
        
        # Initialize direction flags
        direction_flags = {
            "North": 0, "South": 0, "East": 0, "West": 0,
            "NE": 0, "NW": 0, "SE": 0, "SW": 0
        }

        # Check for buildings in each direction
        for b_lat, b_lon in nearby_buildings:
            if b_lat > lat:  # North
                if b_lon > lon:
                    direction_flags["NE"] = 1
                elif b_lon < lon:
                    direction_flags["NW"] = 1
                else:
                    direction_flags["North"] = 1
            elif b_lat < lat:  # South
                if b_lon > lon:
                    direction_flags["SE"] = 1
                elif b_lon < lon:
                    direction_flags["SW"] = 1
                else:
                    direction_flags["South"] = 1
            else:  # Same latitude
                if b_lon > lon:
                    direction_flags["East"] = 1
                elif b_lon < lon:
                    direction_flags["West"] = 1

        return list(direction_flags.values())

    # Process each UHI point in parallel
    results = Parallel(n_jobs=-1)(
        delayed(check_directions)(row['latitude'], row['longitude'])
        for _, row in tqdm(uhi_points.iterrows(), total=len(uhi_points), desc="Processing UHI Points")
    )

    # Convert results to a DataFrame with correct column names
    direction_df = pd.DataFrame(results, columns=["North", "South", "East", "West", "NE", "NW", "SE", "SW"])
    return pd.concat([uhi_points.reset_index(drop=True), direction_df], axis=1)

# File paths
kml_file = "/kaggle/input/ey-challenge/Building_Footprint.kml"

# Parse data
building_centroids = parse_kml(kml_file)
uhi_data = uhi_with_building_counts

# Ensure correct column names
uhi_data.rename(columns={"Latitude": "latitude", "Longitude": "longitude"}, inplace=True)

# Count buildings near each UHI index point (optimized with one-hot directions)
uhi_with_directions = count_buildings_directional(building_centroids, uhi_data)

print(uhi_with_directions.head())


In [ ]:
final_data = combine_two_datasets(final_data, uhi_with_directions)

In [ ]:
final_data.head()

In [ ]:
final_data.columns

In [ ]:
import pandas as pd

# Function to categorize wind direction
def categorize_wind_direction(degrees):
    if (337.5 <= degrees <= 360) or (0 <= degrees < 22.5):
        return 'Wind_N'
    elif 22.5 <= degrees < 67.5:
        return 'Wind_NE'
    elif 67.5 <= degrees < 112.5:
        return 'Wind_E'
    elif 112.5 <= degrees < 157.5:
        return 'Wind_SE'
    elif 157.5 <= degrees < 202.5:
        return 'Wind_S'
    elif 202.5 <= degrees < 247.5:
        return 'Wind_SW'
    elif 247.5 <= degrees < 292.5:
        return 'Wind_W'
    elif 292.5 <= degrees < 337.5:
        return 'Wind_NW'

# Apply function to categorize wind direction
final_data['Wind Direction Category'] = final_data['Wind Direction [degrees]'].apply(categorize_wind_direction)

# One-hot encoding with renamed categories
wind_dummies = pd.get_dummies(final_data['Wind Direction Category'])

# Ensure all 8 wind direction columns exist
for col in ['Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW', 'Wind_W', 'Wind_NW']:
    if col not in wind_dummies:
        wind_dummies[col] = 0  # Add missing columns with 0 values

# Drop original wind direction columns
final_data = final_data.drop(columns=['Wind Direction [degrees]', 'Wind Direction Category'], errors='ignore')

# Merge one-hot encoded columns with the original dataframe
final_data = pd.concat([final_data, wind_dummies], axis=1)

# Reorder columns to maintain consistency
final_data = final_data[['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
                         'B11', 'B12', 'lwir11', 'longitude', 'latitude', 'datetime',
                         'UHI Index', 'Avg Wind Speed [m/s]', 'Solar Flux [W/m^2]',
                         'building_count', 'North', 'South', 'East', 'West', 'NE', 'NW', 'SE',
                         'SW', 'Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW', 'Wind_W', 'Wind_NW']]

# Display final column names
print(final_data.columns)


In [ ]:
# Convert the necessary columns to numeric (if they aren't already)
for col in ['B08', 'B04', 'B03', 'B05', 'B02', 'B11', 'B12']:
    final_data[col] = pd.to_numeric(final_data[col], errors='coerce')

# Define a helper function to compute normalized difference indices
def normalized_difference(band1, band2):
    result = (final_data[band1] - final_data[band2]) / (final_data[band1] + final_data[band2])
    return result.replace([np.inf, -np.inf], np.nan)

# --- Vegetation Indices (example for NDVI, EVI, SAVI) ---
final_data['NDVI'] = normalized_difference('B08', 'B04')  # NDVI
final_data['EVI'] = 2.5 * (final_data['B08'] - final_data['B04']) / (final_data['B08'] + 6 * final_data['B04'] - 7.5 * final_data['B02'] + 1)
final_data['EVI'] = final_data['EVI'].replace([np.inf, -np.inf], np.nan)
final_data['SAVI'] = ((final_data['B08'] - final_data['B04']) / (final_data['B08'] + final_data['B04'] + 0.5)) * 1.5

# --- Updated MSAVI2 Calculation ---
# Compute the temporary Series and force it to be a NumPy array of floats.
temp_series = ( (2 * final_data['B08'] + 1)**2 - 8 * (final_data['B08'] - final_data['B04']) )
temp = np.array(temp_series, dtype=np.float64)
sqrt_temp = np.sqrt(temp)
final_data['MSAVI2'] = (2 * final_data['B08'] + 1 - sqrt_temp) / 2


final_data['GNDVI'] = normalized_difference('B08', 'B03')  # Green NDVI
final_data['RE-NDVI'] = normalized_difference('B08', 'B05')  # Red Edge NDVI
final_data['CIre'] = (final_data['B08'] / final_data['B05']) - 1
final_data['NDRE'] = normalized_difference('B08', 'B05')  # Normalized Difference Red Edge
final_data['GCI'] = (final_data['B08'] / final_data['B03']) - 1
final_data['VARI'] = (final_data['B03'] - final_data['B04']) / (final_data['B03'] + final_data['B04'] - final_data['B02'])
final_data['VARI'] = final_data['VARI'].replace([np.inf, -np.inf], np.nan)

# Water Indices
final_data['NDWI'] = normalized_difference('B03', 'B08')  # Normalized Difference Water Index
final_data['MNDWI'] = normalized_difference('B03', 'B11')  # Modified NDWI
final_data['WRI'] = (final_data['B03'] + final_data['B04']) / (final_data['B08'] + final_data['B11'])
final_data['WRI'] = final_data['WRI'].replace([np.inf, -np.inf], np.nan)
final_data['AWEI'] = 4 * (final_data['B03'] - final_data['B11']) - (0.25 * final_data['B08'] + 2.75 * final_data['B12'])

# Bare Soil Index (BSI)
final_data['BSI'] = (final_data['B11'] + final_data['B04'] - final_data['B08'] - final_data['B02']) / \
                    (final_data['B11'] + final_data['B04'] + final_data['B08'] + final_data['B02'])

# Normalized Burn Ratio (NBR)
final_data['NBR'] = normalized_difference('B08', 'B12')  # (B08 - B12) / (B08 + B12)

# Normalized Burn Ratio 2 (NBR2)
final_data['NBR2'] = normalized_difference('B11', 'B12')  # (B11 - B12) / (B11 + B12)

# Shortwave Infrared Water Stress Index (SIWSI)
final_data['SIWSI'] = normalized_difference('B08', 'B11')  # (B08 - B11) / (B08 + B11)

# Dust Index (DI)
final_data['DI'] = normalized_difference('B11', 'B12')  # (B11 - B12) / (B11 + B12)


# Buildup Index (Normalized Difference Buildup Index)
final_data['NDBI'] = normalized_difference('B11', 'B08')

# Replace any remaining infinities with NaN
final_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print(final_data)


In [ ]:
uhi_data = final_data

In [ ]:
uhi_data.columns

In [ ]:
# Remove duplicate rows from the DataFrame based on specified columns and keep the first occurrence
columns_to_check = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
       'B11', 'B12', 'lwir11', 'UHI Index', 'Avg Wind Speed [m/s]', 'Solar Flux [W/m^2]',
       'building_count', 'North', 'South', 'East', 'West', 'NE', 'NW', 'SE',
       'SW', 'Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW',
       'Wind_W', 'Wind_NW', 'NDVI', 'EVI', 'SAVI', 'MSAVI2', 'GNDVI',
       'RE-NDVI', 'CIre', 'NDRE', 'GCI', 'VARI', 'NDWI', 'MNDWI', 'WRI',
       'AWEI', 'BSI', 'NBR', 'NBR2', 'SIWSI', 'DI', 'NDBI']
for col in columns_to_check:
    # Check if the value is a numpy array and has more than one dimension
    uhi_data[col] = uhi_data[col].apply(lambda x: tuple(x) if isinstance(x, np.ndarray) and x.ndim > 0 else x)

# Now remove duplicates
uhi_data = uhi_data.drop_duplicates(subset=columns_to_check, keep='first')
uhi_data.head()

In [ ]:
# Resetting the index of the dataset
uhi_data=uhi_data.reset_index(drop=True)

In [ ]:
# Retaining only the columns for B01, B06, NDVI, and UHI Index in the dataset.
uhi_data = uhi_data[['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
       'B11', 'B12', 'lwir11',
       'UHI Index', 'Avg Wind Speed [m/s]', 'Solar Flux [W/m^2]',
       'building_count', 'North', 'South', 'East', 'West', 'NE', 'NW', 'SE',
       'SW', 'Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW',
       'Wind_W', 'Wind_NW', 'NDVI', 'EVI', 'SAVI', 'MSAVI2', 'GNDVI',
       'RE-NDVI', 'CIre', 'NDRE', 'GCI', 'VARI', 'NDWI', 'MNDWI', 'WRI',
       'AWEI', 'BSI', 'NBR', 'NBR2', 'SIWSI', 'DI', 'NDBI']]

In [ ]:
# Split the data into features (X) and target (y), and then into training and testing sets
# Ensure column names are strings
X = uhi_data.drop(columns=['UHI Index'])
X.columns = X.columns.astype(str)
y = uhi_data ['UHI Index']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=123)

In [ ]:
X_train.columns = X_train.columns.str.replace(r"[^\w\s]", "", regex=True)
X_test.columns = X_test.columns.str.replace(r"[^\w\s]", "", regex=True)

In [ ]:
y_train = pd.DataFrame(y_train, columns=['UHI Index'])  # Replace 'target' with the actual column name
y_test = pd.DataFrame(y_test, columns=['UHI Index'])

In [ ]:
# Save the feature names before scaling
feature_names = X_train.columns  # Assuming X_train is a DataFrame before scaling

# Scale the training and test data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Convert back to DataFrame with original column names
X_train = pd.DataFrame(X_train, columns=feature_names)
X_test = pd.DataFrame(X_test, columns=feature_names)


In [ ]:
X_train.columns

# Testing Params

In [ ]:
from sklearn.metrics import r2_score

def try_model(search_cv,X_tr, y_tr,X_te,y_te):
    # Fit the RandomizedSearchCV model
    search_cv.fit(X_tr, y_tr)
    
    # Get the best estimator from the search
    best_model = search_cv.best_estimator_
    
    # Best hyperparameters found
    best_hyperparams = search_cv.best_params_
    
    # Make predictions on the training data
    insample_predictions = best_model.predict(X_tr)
    train_r2 = r2_score(y_tr, insample_predictions)
    
    # Make predictions on the test data
    outsample_predictions = best_model.predict(X_te)
    val_r2 = r2_score(y_te, outsample_predictions)
    
    # Extract feature importances if available
    if hasattr(best_model, "feature_importances_"):
        feature_importances = best_model.feature_importances_
        important_features = sorted(zip(X_tr.columns, feature_importances), key=lambda x: x[1], reverse=True)
    else:
        important_features = "Feature importances not available for this model."
    
    return best_hyperparams, important_features, train_r2, val_r2


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.metrics import r2_score

# # Base models
# base_models = [
#     ('RandomForest', RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42)),
#     ('XGBoost', xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=8, random_state=42)),
#     ('GradientBoosting', GradientBoostingRegressor(n_estimators=300, min_samples_split=8, min_samples_leaf=2, max_depth=7, learning_rate=0.1, random_state=42)),
#     ('LightGBM', lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=-1, random_state=42)),
#     ('CatBoost', cb.CatBoostRegressor(iterations=300, learning_rate=0.1, depth=8, verbose=0, random_state=42)),
#     ('KNN', KNeighborsRegressor(weights='distance', n_neighbors=7, metric='manhattan'))
# ]

# # Meta-Learner: XGBoost (You can also try LightGBM or Ridge)
# meta_learner = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)

# # Stacking Regressor
# stacking_regressor = StackingRegressor(estimators=base_models, final_estimator=meta_learner, cv=5)

# --- Combine Training and Test Data ---
X_combined = np.vstack((X_train, X_test))  # Combine X_train and X_test
# Convert y_train and y_test to NumPy arrays
y_combined = np.hstack((np.array(y_train).ravel(), np.array(y_test).ravel()))

# # Train the model on the combined dataset
# stacking_regressor.fit(X_combined, y_combined)

# # --- Evaluate on Combined Data (Optional) ---
# train_r2_combined = r2_score(y_combined, stacking_regressor.predict(X_combined))
# print(f"Final Stacking Regressor R² on Combined Data: {train_r2_combined:.4f}")

# # --- Evaluate on X_test (Validation Performance) ---
# val_r2_final = r2_score(y_test, stacking_regressor.predict(X_test))
# print(f"Final Stacking Regressor Validation R² on X_test: {val_r2_final:.4f}")


In [ ]:
# Model: Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=300, min_samples_split=8, min_samples_leaf=2, max_depth=7, learning_rate=0.1, random_state=42)

# Fit the model
gb_model.fit(X_combined, y_combined)

# Evaluate
val_r2_gb = r2_score(y_test, gb_model.predict(X_test))
print(f"Gradient Boosting Regressor - Validation R²: {val_r2_gb:.4f}")


stacking_regressor = gb_model

In [ ]:
# from sklearn.ensemble import StackingRegressor, VotingRegressor
# from sklearn.linear_model import Ridge
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# import xgboost as xgb
# import lightgbm as lgb
# import catboost as cb
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import r2_score
# import numpy as np

# # Base models
# base_models = [
#     ('RandomForest', RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42)),
#     ('XGBoost', xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=8, random_state=42)),
#     ('GradientBoosting', GradientBoostingRegressor(n_estimators=300, min_samples_split=8, min_samples_leaf=2, max_depth=7, learning_rate=0.1, random_state=42)),
#     ('LightGBM', lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=-1, random_state=42)),
#     ('CatBoost', cb.CatBoostRegressor(iterations=300, learning_rate=0.1, depth=8, verbose=0, random_state=42)),
#     ('KNN', KNeighborsRegressor(weights='distance', n_neighbors=7, metric='manhattan'))
# ]

# # First-layer Stacking Models
# stacking_gb = StackingRegressor(estimators=base_models, final_estimator=GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42), cv=5)
# stacking_lgb = StackingRegressor(estimators=base_models, final_estimator=lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42), cv=5)
# stacking_rf = StackingRegressor(estimators=base_models, final_estimator=RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42), cv=5)

# # Train first-layer stacking models
# stacking_gb.fit(X_train, y_train)
# stacking_lgb.fit(X_train, y_train)
# stacking_rf.fit(X_train, y_train)

# # Generate meta-features
# train_meta_features = np.column_stack([
#     stacking_gb.predict(X_train),
#     stacking_lgb.predict(X_train),
#     stacking_rf.predict(X_train)
# ])

# test_meta_features = np.column_stack([
#     stacking_gb.predict(X_test),
#     stacking_lgb.predict(X_test),
#     stacking_rf.predict(X_test)
# ])

# voting_regressor = VotingRegressor([
#     ('GB_Stacked', stacking_gb),
#     ('LGB_Stacked', stacking_lgb),
#     ('RF_Stacked', stacking_rf)
# ], weights=[0.4, 0.35, 0.25])  # Example: Adjust weights based on validation scores


# # Train final voting regressor
# voting_regressor.fit(X_train, y_train)

# # Evaluate
# train_r2 = r2_score(y_train, voting_regressor.predict(X_train))
# val_r2 = r2_score(y_test, voting_regressor.predict(X_test))

# print(f"Final Voting Stacking Model Train R²: {train_r2:.4f}")
# print(f"Final Voting Stacking Model Validation R²: {val_r2:.4f}")


In [ ]:
# Load the training data from csv file and display the first few rows to inspect the data
sub_df = pd.read_csv("/kaggle/input/submission/Submission_template_UHI2025-v2.csv")
sub_df.head()

In [ ]:
# Mapping satellite data with training data.
sub_final_data = map_satellite_data('/kaggle/input/ey-challenge/S2_sample.tiff', '/kaggle/input/submission/Submission_template_UHI2025-v2.csv')

In [ ]:
# Mapping satellite data with training data.
sub_final_data2 = map_satellite_data2('/kaggle/input/landsat-tiff1/Landsat_LST.tiff', "/kaggle/input/submission/Submission_template_UHI2025-v2.csv")

In [ ]:
sub_final_data = combine_two_datasets(sub_final_data, sub_final_data2)

In [ ]:
weather_data_path = '/kaggle/input/ey-challenge/NY_Mesonet_Weather.xlsx'
# Load weather data from both sheets
bronx_weather = pd.read_excel(weather_data_path, sheet_name="Bronx")
manhattan_weather = pd.read_excel(weather_data_path, sheet_name="Manhattan")

In [ ]:
train_path = "/kaggle/input/ey-opendata/Training_data_uhi_index_2025-02-18.csv"
ground_df = pd.read_csv(train_path)
ground_df['datetime'] = pd.to_datetime(ground_df['datetime'], format='%d-%m-%Y %H:%M')
n = len(sub_df)  # Get the number of rows in submission
sub_df['datetime'] = ground_df['datetime'].iloc[:n].values

In [ ]:
from datetime import datetime
from geopy.distance import geodesic

# Define weather station locations
bronx_coords = (40.87248, -73.89352)
manhattan_coords = (40.76754, -73.96449)

# Convert datetime columns to datetime objects
sub_df['datetime'] = pd.to_datetime(sub_df['datetime'], format='%d-%m-%Y %H:%M')
bronx_weather['Date / Time'] = pd.to_datetime(bronx_weather['Date / Time'])
manhattan_weather['Date / Time'] = pd.to_datetime(manhattan_weather['Date / Time'])

# Function to determine the closest weather station
def get_closest_station(lat, lon):
    bronx_dist = geodesic((lat, lon), bronx_coords).km
    manhattan_dist = geodesic((lat, lon), manhattan_coords).km
    return 'bronx' if bronx_dist < manhattan_dist else 'manhattan'

# Function to find the closest weather record
def get_closest_weather_data(row):
    station = get_closest_station(row['Latitude'], row['Longitude'])
    weather_df = bronx_weather if station == 'bronx' else manhattan_weather
    
    closest_time_idx = (weather_df['Date / Time'] - row['datetime']).abs().idxmin()
    closest_weather = weather_df.loc[closest_time_idx, ['Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']]
    return closest_weather

# Apply function to df1
sub_df[['Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']] = ground_df.apply(get_closest_weather_data, axis=1)

# Display updated df1
sub_df.head()


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Function to parse KML and extract building centroids
def parse_kml(file_path):
    namespace = {"kml": "http://www.opengis.net/kml/2.2"}
    tree = ET.parse(file_path)
    root = tree.getroot()

    placemarks = root.findall(".//kml:Placemark", namespace)
    building_centroids = []

    for placemark in placemarks:
        coords_element = placemark.find(".//kml:coordinates", namespace)
        if coords_element is not None:
            coords_text = coords_element.text.strip()
            coords_list = [
                tuple(map(float, coord.split(",")[:2]))  # Extract lon, lat only
                for coord in coords_text.split()
            ]
            # Compute centroid
            centroid_lat = sum(p[1] for p in coords_list) / len(coords_list)
            centroid_lon = sum(p[0] for p in coords_list) / len(coords_list)
            building_centroids.append((centroid_lat, centroid_lon))

    return np.array(building_centroids)  # Convert to NumPy array for speed

# Fast Haversine distance calculation
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c  # Distance in meters

# Function to count buildings near UHI points (Parallelized)
def count_buildings_near_uhi(building_centroids, uhi_points, radius=80):
    def count_nearby_buildings(lat, lon):
        distances = haversine(lat, lon, building_centroids[:, 0], building_centroids[:, 1])
        return np.sum(distances <= radius)  # Count buildings within radius

    results = Parallel(n_jobs=-1)(
        delayed(count_nearby_buildings)(row['latitude'], row['longitude'])
        for _, row in tqdm(uhi_points.iterrows(), total=len(uhi_points), desc="Processing UHI Points")
    )

    uhi_points['building_count'] = results
    return uhi_points

# File paths
kml_file = "/kaggle/input/ey-challenge/Building_Footprint.kml"

# Parse data
building_centroids = parse_kml(kml_file)
sub_uhi_data = sub_df

# Ensure correct column names
sub_uhi_data.rename(columns={"Latitude": "latitude", "Longitude": "longitude"}, inplace=True)

# Count buildings near each UHI index point (optimized)
sub_uhi_with_building_counts = count_buildings_near_uhi(building_centroids, sub_uhi_data)

print(sub_uhi_with_building_counts.head())


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Function to parse KML and extract building centroids
def parse_kml(file_path):
    namespace = {"kml": "http://www.opengis.net/kml/2.2"}
    tree = ET.parse(file_path)
    root = tree.getroot()

    placemarks = root.findall(".//kml:Placemark", namespace)
    building_centroids = []

    for placemark in placemarks:
        coords_element = placemark.find(".//kml:coordinates", namespace)
        if coords_element is not None:
            coords_text = coords_element.text.strip()
            coords_list = [
                tuple(map(float, coord.split(",")[:2]))  # Extract lon, lat only
                for coord in coords_text.split()
            ]
            # Compute centroid
            centroid_lat = sum(p[1] for p in coords_list) / len(coords_list)
            centroid_lon = sum(p[0] for p in coords_list) / len(coords_list)
            building_centroids.append((centroid_lat, centroid_lon))

    return np.array(building_centroids)  # Convert to NumPy array for speed

# Fast Haversine distance calculation
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c  # Distance in meters

# Function to count buildings in directional quadrants
def count_buildings_directional(building_centroids, uhi_points, radius=80):
    def check_directions(lat, lon):
        distances = haversine(lat, lon, building_centroids[:, 0], building_centroids[:, 1])
        nearby_buildings = building_centroids[distances <= radius]  # Filter buildings within the radius
        
        # Initialize direction flags
        direction_flags = {
            "North": 0, "South": 0, "East": 0, "West": 0,
            "NE": 0, "NW": 0, "SE": 0, "SW": 0
        }

        # Check for buildings in each direction
        for b_lat, b_lon in nearby_buildings:
            if b_lat > lat:  # North
                if b_lon > lon:
                    direction_flags["NE"] = 1
                elif b_lon < lon:
                    direction_flags["NW"] = 1
                else:
                    direction_flags["North"] = 1
            elif b_lat < lat:  # South
                if b_lon > lon:
                    direction_flags["SE"] = 1
                elif b_lon < lon:
                    direction_flags["SW"] = 1
                else:
                    direction_flags["South"] = 1
            else:  # Same latitude
                if b_lon > lon:
                    direction_flags["East"] = 1
                elif b_lon < lon:
                    direction_flags["West"] = 1

        return list(direction_flags.values())

    # Process each UHI point in parallel
    results = Parallel(n_jobs=-1)(
        delayed(check_directions)(row['latitude'], row['longitude'])
        for _, row in tqdm(uhi_points.iterrows(), total=len(uhi_points), desc="Processing UHI Points")
    )

    # Convert results to a DataFrame with correct column names
    direction_df = pd.DataFrame(results, columns=["North", "South", "East", "West", "NE", "NW", "SE", "SW"])
    return pd.concat([uhi_points.reset_index(drop=True), direction_df], axis=1)

# File paths
kml_file = "/kaggle/input/ey-challenge/Building_Footprint.kml"

# Parse data
building_centroids = parse_kml(kml_file)
sub_uhi_data = sub_uhi_with_building_counts

# Ensure correct column names
sub_uhi_data.rename(columns={"Latitude": "latitude", "Longitude": "longitude"}, inplace=True)

# Count buildings near each UHI index point (optimized with one-hot directions)
sub_uhi_with_directions = count_buildings_directional(building_centroids, sub_uhi_data)

print(sub_uhi_with_directions.head())


In [ ]:
sub_final_data = combine_two_datasets(sub_final_data, sub_uhi_with_directions)

In [ ]:
import pandas as pd

# Function to categorize wind direction
def categorize_wind_direction(degrees):
    if (337.5 <= degrees <= 360) or (0 <= degrees < 22.5):
        return 'Wind_N'
    elif 22.5 <= degrees < 67.5:
        return 'Wind_NE'
    elif 67.5 <= degrees < 112.5:
        return 'Wind_E'
    elif 112.5 <= degrees < 157.5:
        return 'Wind_SE'
    elif 157.5 <= degrees < 202.5:
        return 'Wind_S'
    elif 202.5 <= degrees < 247.5:
        return 'Wind_SW'
    elif 247.5 <= degrees < 292.5:
        return 'Wind_W'
    elif 292.5 <= degrees < 337.5:
        return 'Wind_NW'

# Apply function to categorize wind direction
sub_final_data['Wind Direction Category'] = sub_final_data['Wind Direction [degrees]'].apply(categorize_wind_direction)

# One-hot encoding with renamed categories
sub_wind_dummies = pd.get_dummies(sub_final_data['Wind Direction Category'])

# Ensure all 8 wind direction columns exist
for col in ['Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW', 'Wind_W', 'Wind_NW']:
    if col not in sub_wind_dummies:
        sub_wind_dummies[col] = 0  # Add missing columns with 0 values

# Drop original wind direction columns
sub_final_data = sub_final_data.drop(columns=['Wind Direction [degrees]', 'Wind Direction Category'], errors='ignore')

# Merge one-hot encoded columns with the original dataframe
sub_final_data = pd.concat([sub_final_data, sub_wind_dummies], axis=1)

# Reorder columns to maintain consistency
sub_final_data = sub_final_data[['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
                         'B11', 'B12', 'lwir11', 'longitude', 'latitude', 'datetime',
                         'UHI Index', 'Avg Wind Speed [m/s]', 'Solar Flux [W/m^2]',
                         'building_count', 'North', 'South', 'East', 'West', 'NE', 'NW', 'SE',
                         'SW', 'Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW', 'Wind_W', 'Wind_NW']]

# Display final column names
print(sub_final_data.columns)


In [ ]:
import pandas as pd
import numpy as np

# Convert the necessary columns to numeric (if they aren't already)
for col in ['B08', 'B04', 'B03', 'B05', 'B02', 'B11', 'B12']:
    sub_final_data[col] = pd.to_numeric(sub_final_data[col], errors='coerce')

# Define a helper function to compute normalized difference indices
def normalized_difference(band1, band2):
    result = (sub_final_data[band1] - sub_final_data[band2]) / (sub_final_data[band1] + sub_final_data[band2])
    return result.replace([np.inf, -np.inf], np.nan)

# --- Vegetation Indices (example for NDVI, EVI, SAVI) ---
sub_final_data['NDVI'] = normalized_difference('B08', 'B04')  # NDVI
sub_final_data['EVI'] = 2.5 * (sub_final_data['B08'] - sub_final_data['B04']) / (
    sub_final_data['B08'] + 6 * sub_final_data['B04'] - 7.5 * sub_final_data['B02'] + 1)
sub_final_data['EVI'] = sub_final_data['EVI'].replace([np.inf, -np.inf], np.nan)
sub_final_data['SAVI'] = ((sub_final_data['B08'] - sub_final_data['B04']) / 
                         (sub_final_data['B08'] + sub_final_data['B04'] + 0.5)) * 1.5

# --- Updated MSAVI2 Calculation ---
# Compute the temporary Series and force it to be a NumPy array of floats.
temp_series = ((2 * sub_final_data['B08'] + 1)**2 - 8 * (sub_final_data['B08'] - sub_final_data['B04']))
temp = np.array(temp_series, dtype=np.float64)
sqrt_temp = np.sqrt(temp)
sub_final_data['MSAVI2'] = (2 * sub_final_data['B08'] + 1 - sqrt_temp) / 2

sub_final_data['GNDVI'] = normalized_difference('B08', 'B03')  # Green NDVI
sub_final_data['RE-NDVI'] = normalized_difference('B08', 'B05')  # Red Edge NDVI
sub_final_data['CIre'] = (sub_final_data['B08'] / sub_final_data['B05']) - 1
sub_final_data['NDRE'] = normalized_difference('B08', 'B05')  # Normalized Difference Red Edge
sub_final_data['GCI'] = (sub_final_data['B08'] / sub_final_data['B03']) - 1
sub_final_data['VARI'] = (sub_final_data['B03'] - sub_final_data['B04']) / (
    sub_final_data['B03'] + sub_final_data['B04'] - sub_final_data['B02'])
sub_final_data['VARI'] = sub_final_data['VARI'].replace([np.inf, -np.inf], np.nan)

# --- Water Indices ---
sub_final_data['NDWI'] = normalized_difference('B03', 'B08')  # Normalized Difference Water Index
sub_final_data['MNDWI'] = normalized_difference('B03', 'B11')  # Modified NDWI
sub_final_data['WRI'] = (sub_final_data['B03'] + sub_final_data['B04']) / (
    sub_final_data['B08'] + sub_final_data['B11'])
sub_final_data['WRI'] = sub_final_data['WRI'].replace([np.inf, -np.inf], np.nan)
sub_final_data['AWEI'] = 4 * (sub_final_data['B03'] - sub_final_data['B11']) - (
    0.25 * sub_final_data['B08'] + 2.75 * sub_final_data['B12'])

# --- Bare Soil Index (BSI) ---
sub_final_data['BSI'] = (sub_final_data['B11'] + sub_final_data['B04'] - sub_final_data['B08'] - sub_final_data['B02']) / \
                        (sub_final_data['B11'] + sub_final_data['B04'] + sub_final_data['B08'] + sub_final_data['B02'])

# --- Normalized Burn Ratio (NBR) ---
sub_final_data['NBR'] = normalized_difference('B08', 'B12')  # (B08 - B12) / (B08 + B12)

# --- Normalized Burn Ratio 2 (NBR2) ---
sub_final_data['NBR2'] = normalized_difference('B11', 'B12')  # (B11 - B12) / (B11 + B12)

# --- Shortwave Infrared Water Stress Index (SIWSI) ---
sub_final_data['SIWSI'] = normalized_difference('B08', 'B11')  # (B08 - B11) / (B08 + B11)

# --- Dust Index (DI) ---
sub_final_data['DI'] = normalized_difference('B11', 'B12')  # (B11 - B12) / (B11 + B12)

# --- Buildup Index (Normalized Difference Buildup Index) ---
sub_final_data['NDBI'] = normalized_difference('B11', 'B08')

# Replace any remaining infinities with NaN
sub_final_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print(sub_final_data)


In [ ]:
sub_uhi_data = sub_final_data

In [ ]:
# Resetting the index of the dataset
sub_uhi_data=sub_uhi_data.reset_index(drop=True)

In [ ]:
# Retaining only the columns for B01, B06, NDVI, and UHI Index in the dataset.
sub_uhi_data = sub_uhi_data[['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
       'B11', 'B12', 'lwir11',
       'UHI Index', 'Avg Wind Speed [m/s]', 'Solar Flux [W/m^2]',
       'building_count', 'North', 'South', 'East', 'West', 'NE', 'NW', 'SE',
       'SW', 'Wind_N', 'Wind_NE', 'Wind_E', 'Wind_SE', 'Wind_S', 'Wind_SW',
       'Wind_W', 'Wind_NW', 'NDVI', 'EVI', 'SAVI', 'MSAVI2', 'GNDVI',
       'RE-NDVI', 'CIre', 'NDRE', 'GCI', 'VARI', 'NDWI', 'MNDWI', 'WRI',
       'AWEI', 'BSI', 'NBR', 'NBR2', 'SIWSI', 'DI', 'NDBI']]

In [ ]:
# Split the data into features (X) and target (y), and then into training and testing sets
# Ensure column names are strings
sub_X = sub_uhi_data.drop(columns=['UHI Index'])
sub_X.columns = sub_X.columns.astype(str)


In [ ]:
sub_X.columns = sub_X.columns.str.replace(r"[^\w\s]", "", regex=True)

In [ ]:
feature_names = sub_X.columns 
sub_X = sc.transform(sub_X)

In [ ]:
sub_X = pd.DataFrame(sub_X, columns=feature_names)

In [ ]:
sub_y = stacking_regressor.predict(sub_X)

In [ ]:
sub_2 = pd.read_csv('/kaggle/input/submission/Submission_template_UHI2025-v2.csv')
sub_2['UHI Index'] = sub_y

In [ ]:
# Save the updated submission file
sub_2.to_csv('final_submission.csv', index=False)

print("Submission file with datetime column is ready! 🚀")